# Objetivo 1 · Entrenamiento real de nnU-Net: 100 épocas por fold (Colab)
**Reto CIMAT / BraTS 2018**

Por fold: **~1.8 h en una A100 con RAM amplia** (65 s por época, medido en el fold 0) o ~13 h en una T4. `FOLDS` define cuáles se entrenan, uno tras otro; los folds 1 a 4 son ~7.5 h en la A100, más que una sesión de Colab gratis (~12 h, y menos si el navegador queda inactivo). Por eso este notebook está hecho para **ejecutarse completo en cada sesión**:

- **Primera sesión:** instala, preprocesa (30–60 min), guarda el preprocesado en Drive si cabe, y empieza a entrenar.
- **Sesiones siguientes:** recupera el preprocesado de Drive y **retoma solo desde el último checkpoint**.
- Guarda un checkpoint **cada 5 épocas** (~40 min); nnU-Net por defecto lo hace cada 50 (~6 h). Si la sesión se corta, se pierden como mucho unos 40 minutos.
- Cuando el entrenamiento termina, volver a ejecutarlo no repite nada: carga el modelo final y solo hace la validación.

**Cada sesión:** *Entorno de ejecución → Cambiar tipo → GPU A100 con RAM amplia* (o T4) y luego *Entorno de ejecución → Ejecutar todas*. Deja la pestaña abierta: Colab gratis corta las sesiones sin actividad en el navegador.

Requisito: `Dataset501_BraTS2018.zip` en `MyDrive/reto_cimat/` (el mismo de la prueba).

## 0. Parámetros

In [ ]:
FOLDS = [1, 2, 3, 4]                       # se entrenan en orden; los terminados se saltan (el 0 ya esta)
TRAINER = 'nnUNetTrainer_100epochs_ckpt5'   # 100 epocas + checkpoint cada 5 (se define en la seccion 3)
GUARDAR_PREPROCESADO = True                 # copia el preprocesado a Drive si hay espacio
LIBERAR_AL_TERMINAR = True                 # desconecta la GPU al terminar: deja de gastar unidades
DRIVE = '/content/drive/MyDrive/reto_cimat'

## 1. GPU y Drive

In [ ]:
import os, json, shutil, subprocess, time
import torch
from google.colab import drive
assert torch.cuda.is_available(), 'No hay GPU: cambia el tipo de entorno a GPU'
props = torch.cuda.get_device_properties(0)
import psutil
print(f'GPU: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB  | CPUs: {os.cpu_count()}  '
      f'| RAM: {psutil.virtual_memory().total / 1e9:.0f} GB')
if os.cpu_count() <= 2:
    print('AVISO: solo 2 CPUs; activa "RAM amplia" para tener mas nucleos y entrenar mas rapido')
drive.mount('/content/drive')
ZIP = f'{DRIVE}/Dataset501_BraTS2018.zip'
assert os.path.exists(ZIP), f'No encuentro {ZIP}'

## 2. nnU-Net y variables de entorno

In [ ]:
!pip install -q nnunetv2
import importlib.metadata as md
print('nnunetv2', md.version('nnunetv2'))

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = f'{DRIVE}/nnUNet_results'
os.environ['nnUNet_n_proc_DA'] = str(os.cpu_count())
os.environ['nnUNet_compile'] = 'f'          # misma configuracion que la prueba (D23)
for k in ('nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results'):
    os.makedirs(os.environ[k], exist_ok=True)

DS = 'Dataset501_BraTS2018'
RAW, PRE = f"{os.environ['nnUNet_raw']}/{DS}", f"{os.environ['nnUNet_preprocessed']}/{DS}"
salida = lambda fold: f"{os.environ['nnUNet_results']}/{DS}/{TRAINER}__nnUNetPlans__3d_fullres/fold_{fold}"
CACHE = f'{DRIVE}/nnUNet_preprocessed_{DS}.tar'
DIR_RAW, DIR_PRE = os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed']

## 3. Entrenador con checkpoint cada 5 épocas
Hereda de `nnUNetTrainer_100epochs`, que ya trae nnU-Net, y solo cambia la frecuencia de guardado. No redefine `__init__`: nnU-Net guarda los argumentos del constructor en el checkpoint y un constructor distinto rompe la reanudación. El archivo se escribe dentro del paquete instalado, donde nnU-Net busca los entrenadores, así que hay que repetirlo en cada sesión (esta celda lo hace).

In [ ]:
import nnunetv2
codigo = '''from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs


class nnUNetTrainer_100epochs_ckpt5(nnUNetTrainer_100epochs):
    """100 epocas; guarda checkpoint_latest cada 5 epocas (por defecto, cada 50)."""

    def on_train_start(self):
        super().on_train_start()
        self.save_every = 5
'''
destino = os.path.join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer', 'variants', f'{TRAINER}.py')
open(destino, 'w').write(codigo)

# Comprobacion: nnU-Net debe encontrarlo igual que lo hara el comando de entrenamiento
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
clase = recursive_find_python_class(os.path.join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer'),
                                    TRAINER, 'nnunetv2.training.nnUNetTrainer')
assert clase is not None, 'nnU-Net no encuentra el entrenador'
print('entrenador listo:', clase)

## 4. Dataset crudo (Drive → disco local)

In [ ]:
%%time
if not os.path.exists(f'{RAW}/dataset.json'):
    !cp "{ZIP}" /content/ && unzip -q -o /content/Dataset501_BraTS2018.zip -d {DIR_RAW} && rm /content/Dataset501_BraTS2018.zip
print(len(os.listdir(f'{RAW}/imagesTr')), 'imagenes')   # esperado: 1140

## 5. Preprocesado: recuperarlo de Drive o generarlo
La primera vez se genera (30–60 min) y, si hay espacio, se guarda en Drive como `.tar`. Las siguientes sesiones solo lo copian y descomprimen.

In [ ]:
%%time
import psutil
if os.path.exists(f'{PRE}/nnUNetPlans.json'):
    print('preprocesado ya presente en esta sesion')
elif os.path.exists(CACHE):
    print('recuperando preprocesado desde Drive...')
    !tar -xf "{CACHE}" -C {DIR_PRE}
else:
    # ~3 GB de RAM por proceso: 2 en Colab normal, hasta 8 con RAM amplia
    NP = max(2, min(8, os.cpu_count(), int(psutil.virtual_memory().total / 3e9)))
    print('procesos de preprocesamiento:', NP)
    !nnUNetv2_plan_and_preprocess -d 501 -c 3d_fullres --verify_dataset_integrity -np {NP}
    if GUARDAR_PREPROCESADO:
        tam = int(subprocess.check_output(['du', '-sb', PRE]).split()[0])
        libre = shutil.disk_usage('/content/drive/MyDrive').free
        print(f'preprocesado: {tam / 1e9:.1f} GB | libre en Drive: {libre / 1e9:.1f} GB')
        if tam < libre - 2e9:
            !tar -cf "{CACHE}" -C {DIR_PRE} {DS}
            print('guardado en', CACHE)
        else:
            print('no cabe en Drive: la proxima sesion volvera a preprocesar')

# Siempre nuestros folds (D22), nunca los aleatorios de nnU-Net
shutil.copy(f'{RAW}/splits_final.json', f'{PRE}/splits_final.json')
splits = json.load(open(f'{PRE}/splits_final.json'))
assert [len(s['val']) for s in splits] == [57] * 5
print('folds ok | se entrenaran:', FOLDS)

## 6. Estado de cada fold
Muestra, para cada fold de `FOLDS`, si se empieza de cero, se retoma o ya terminó.

In [ ]:
import glob, re

def progreso(fold):
    """Epocas completadas, tiempos por epoca y ultimo pseudo-Dice [WT, TC, ET], leyendo todos los logs."""
    texto = ''.join(open(f).read() for f in sorted(glob.glob(f'{salida(fold)}/training_log_*.txt')))
    tiempos = [float(t) for t in re.findall(r'Epoch time: ([\d.]+) s', texto)]
    dice = [[float(x) for x in re.findall(r'\d+\.\d+', d)] for d in re.findall(r'Pseudo dice \[(.*?)\]', texto)]
    return len(tiempos), tiempos, dice[-1] if dice else None


def curva(fold):
    """Pseudo-Dice [WT, TC, ET] por epoca (si una epoca se repitio al retomar, vale la ultima)."""
    por_epoca, epoca = {}, None
    for f in sorted(glob.glob(f'{salida(fold)}/training_log_*.txt')):
        for linea in open(f):
            m = re.search(r'Epoch (\d+)\s*$', linea)
            if m:
                epoca = int(m.group(1))
            m = re.search(r'Pseudo dice \[(.*?)\]', linea)
            if m and epoca is not None:
                por_epoca[epoca] = [float(x) for x in re.findall(r'\d+\.\d+', m.group(1))]
    return por_epoca


def estado(fold):
    s = salida(fold)
    if os.path.exists(f'{s}/checkpoint_final.pth'):
        return 'terminado' + ('' if os.path.exists(f'{s}/validation/summary.json') else ' (falta validar)')
    if os.path.exists(f'{s}/checkpoint_latest.pth'):
        ck = torch.load(f'{s}/checkpoint_latest.pth', map_location='cpu', weights_only=False)
        return f"se retoma desde la epoca {ck['current_epoch']} de 100"
    return 'se empieza de cero'

for f in FOLDS:
    print(f'fold {f}: {estado(f)}')

## 7. Entrenar los folds de `FOLDS`, uno tras otro
`--c` retoma desde el último checkpoint, o empieza de cero si no hay ninguno. Mientras entrena se registra el uso de la GPU en Drive (`gpu_uso_fold<k>.csv`) para confirmar si la CPU es el cuello de botella (D23).
Sin `--npz`: las probabilidades de validación ocuparían varios GB de Drive y solo sirven para ensamblar configuraciones, que no vamos a hacer.

In [ ]:
for fold in FOLDS:
    if estado(fold) == 'terminado':
        print(f'fold {fold}: terminado, se salta')
        continue
    print(f'\n=== fold {fold}: {estado(fold)} ===')
    gpu_log = f'{DRIVE}/gpu_uso_fold{fold}.csv'
    monitor = subprocess.Popen(
        f'nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used --format=csv,noheader -l 60 >> "{gpu_log}"',
        shell=True)
    try:
        !nnUNetv2_train 501 3d_fullres {fold} -tr {TRAINER} --c
    finally:
        monitor.terminate()

## 8. Resumen por fold
Se puede ejecutar en cualquier momento, incluso en otra sesión. Incluye el fold 0 como referencia.

In [ ]:
import numpy as np, pandas as pd
filas = []
for fold in sorted(set(FOLDS) | {0}):
    hechas, tiempos, ultimo = progreso(fold)
    fila = {'fold': fold, 'estado': estado(fold), 'epocas': hechas,
            's/epoca': round(float(np.median(tiempos))) if tiempos else None}
    gpu_log = f'{DRIVE}/gpu_uso_fold{fold}.csv'
    if os.path.exists(gpu_log):
        uso = pd.read_csv(gpu_log, header=None, names=['hora', 'gpu', 'mem'])
        fila['uso GPU %'] = round(uso['gpu'].str.replace('%', '').astype(float).median())
    resumen = f'{salida(fold)}/validation/summary.json'
    if os.path.exists(resumen):
        m = json.load(open(resumen))['mean']
        for k, n in (('(1, 2, 3)', 'WT'), ('(2, 3)', 'TC'), ('(3,)', 'ET')):
            fila[f'Dice {n}'] = round(m[k]['Dice'], 3)
    filas.append(fila)
tabla = pd.DataFrame(filas).set_index('fold')
print(tabla.to_string())
validados = tabla.filter(like='Dice').dropna()
if len(validados) > 1:
    print(f'\nmedia de {len(validados)} folds:', validados.mean().round(3).to_dict())
print('\nPara ver un fold: visor_colab.ipynb con MODELO =', repr(TRAINER), 'y FOLD = k')

## 9. Liberar la GPU
Si **todos** los folds de `FOLDS` terminaron y `LIBERAR_AL_TERMINAR` es `True`, desconecta el entorno para no gastar unidades. Si alguno quedó a medias, no desconecta: vuelve a ejecutar todo para retomar.

In [ ]:
if LIBERAR_AL_TERMINAR and all(estado(f) == 'terminado' for f in FOLDS):
    print('todos los folds terminados y validados: liberando la GPU')
    from google.colab import runtime
    runtime.unassign()
else:
    print('no se libera: algun fold no termino (o LIBERAR_AL_TERMINAR = False)')